# Modelo 2 — Sistema **SEMAJV**
## Análisis del máximo acercamiento del asteroide (99942) Apophis a la Tierra — abril 2029

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Abril 2026  

---

> **SEMAJV** = **S**ol · **E**arth (Tierra) · **M**oon (Luna) · **A**pophis · **J**upiter · **V**enus
>
> Este cuaderno implementa la integración numérica de N cuerpos del sistema SEMAJV usando `pymcel`
> con el objetivo de determinar la **fecha exacta y la distancia mínima** del acercamiento histórico
> de Apophis a la Tierra en abril de 2029.


## Contexto científico

El asteroide **(99942) Apophis** (diámetro ≈ 370 m, masa ≈ 2.7 × 10¹⁰ kg) fue descubierto en junio de 2004.
Durante varios años estuvo clasificado como una amenaza de impacto de nivel 4 en la escala de Torino,
hasta que las observaciones adicionales descartaron el impacto en 2029.  
No obstante, el asteroide experimentará un **acercamiento histórico a ~38 000 km** de la superficie terrestre
(< 10 radios terrestres; inferior a la órbita de los satélites geoestacionarios) el **13 de abril de 2029**.

### Metodología

1. **Condiciones iniciales**: obtenidas de NASA Horizons el 2028-01-01 referidas al baricentro del sistema solar.
2. **Integración N-cuerpos**: `pymcel.ncuerpos_solucion()` resuelve las ecuaciones de Newton con G = 1 (unidades canónicas).
3. **Unidades canónicas**: AU · M☉ · UT, donde UT = √(AU³/GM☉) ≈ 58.12 días ≈ año/2π.
4. **Refinamiento**: segunda integración con paso fino (≈ minutos) en la ventana del encuentro.
5. **Validación**: comprobación de conservación de energía y momento angular.


In [ ]:
%pip install -Uq pymcel


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime, timedelta

import pymcel as pc
from pymcel import constantes as const

print('Todas las librerías cargadas correctamente.')


## 1. Unidades canónicas

Adoptamos el sistema de unidades astronómicas estándar en el que **G = 1**:

| Magnitud | Símbolo | Valor SI |
|----------|---------|----------|
| Longitud | 1 AU | 1.496 × 10⁸ km |
| Masa     | 1 M☉ | 1.989 × 10³⁰ kg |
| Tiempo   | UT = √(AU³/GM☉) | ≈ 58.12 días ≈ año/2π |
| Velocidad| AU/UT | ≈ 29.79 km/s |

La masa canónica de cada cuerpo es simplemente la razón de masas respecto al Sol:
$m_i^{\rm can} = GM_i / GM_\odot = M_i / M_\odot$.
El período orbital de la Tierra en estas unidades es $2\pi \approx 6.283$ UT.


In [ ]:
# ── Constantes físicas ──────────────────────────────────────────────────────
AU_km    = 149_597_870.7        # km por AU
AU_m     = AU_km * 1e3          # m por AU
M_sun_kg = 1.989e30             # kg (masa solar)
G_SI     = 6.674e-11            # m³ kg⁻¹ s⁻²
R_Earth  = 6_371.0              # km (radio medio terrestre)

# Parámetros gravitacionales GM (km³/s²) — valores JPL DE441
GM_sun_km3s2     = 1.32712440018e11
GM_earth_km3s2   = 3.986004418e5
GM_moon_km3s2    = 4.9048695e3
GM_jupiter_km3s2 = 1.26686534e8
GM_venus_km3s2   = 3.24858592e5
M_apophis_kg     = 2.7e10          # ~370 m, densidad ~1.9 g/cm³

# ── Unidad de tiempo canónica UT = sqrt(AU³ / G·M_sun) ──────────────────────
UT_s    = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))   # segundos
UT_days = UT_s / 86_400.0                          # días
UT_yr   = UT_days / 365.25                         # años

# Unidad de velocidad: AU/UT (km/s)
vel_unit = AU_km / UT_s    # ≈ 29.79 km/s

# ── Masas canónicas (adimensionales) m_i = GM_i / GM_sun ────────────────────
m_sol     = 1.0
m_tierra  = GM_earth_km3s2   / GM_sun_km3s2   # ≈ 3.003e-6
m_luna    = GM_moon_km3s2    / GM_sun_km3s2   # ≈ 3.694e-8
m_jupiter = GM_jupiter_km3s2 / GM_sun_km3s2   # ≈ 9.548e-4
m_venus   = GM_venus_km3s2   / GM_sun_km3s2   # ≈ 2.448e-6
m_apophis = M_apophis_kg     / M_sun_kg        # ≈ 1.36e-20 (despreciable)

print('=' * 58)
print(f'Unidad de tiempo:     UT = {UT_days:.4f} días = {UT_yr:.6f} años')
print(f'Unidad de velocidad:  1 AU/UT = {vel_unit:.4f} km/s')
print(f'Período Tierra:       2π UT = {2*np.pi*UT_days:.2f} días  (≈ 365.25) ✓')
print('=' * 58)
print('Masas canónicas (G=1):')
print(f'  Sol:      {m_sol:.6f}')
print(f'  Tierra:   {m_tierra:.6e}')
print(f'  Luna:     {m_luna:.6e}')
print(f'  Júpiter:  {m_jupiter:.6e}')
print(f'  Venus:    {m_venus:.6e}')
print(f'  Apophis:  {m_apophis:.6e}  (negligible)')


## 2. Condiciones iniciales vía NASA Horizons

Usamos `pc.consulta_horizons()` para obtener las posiciones y velocidades de cada cuerpo
referidas al **baricentro del sistema solar** (`location='@0'`) en la época **2028-01-01** ($t_0$).

- Las posiciones se devuelven en **AU** (baricéntricas).
- Las velocidades se devuelven en **AU/día**.

Convertimos las velocidades a unidades canónicas (AU/UT) multiplicando por $\text{UT}_{\rm días} \approx 58.12$.


In [ ]:
T0_STR  = '2028-01-01'   # época inicial
t0_date = datetime(2028, 1, 1)

# IDs en NASA Horizons
cuerpos_hz = [
    ('Sol',     '10'),
    ('Tierra',  '399'),
    ('Luna',    '301'),
    ('Jupiter', '599'),
    ('Venus',   '299'),
    ('Apophis', '99942'),
]

# Consulta de un único punto temporal por cuerpo
estados_raw = {}
for nombre, id_hz in cuerpos_hz:
    print(f'Consultando {nombre:8s} (ID={id_hz})...', end=' ')
    df = pc.consulta_horizons(
        id          = id_hz,
        location    = '@0',         # baricentro del sistema solar
        epochs      = T0_STR,
        datos       = 'vectors',
        propiedades = 'default',
    )
    # Access the first element of the tuple which is the Table object
    estados_raw[nombre] = df[0]
    print(f'OK  ({len(df[0])} fila | cols: {list(df[0].columns)[:6]})')

print('\n✓ Efemérides descargadas correctamente.')


In [ ]:
def extraer_estado_canonico(df):
    """
    Extrae r [AU] y v [AU/UT] de la primera fila del DataFrame de Horizons.
    Horizons (astroquery vectors) devuelve posiciones en AU y velocidades en AU/día.
    Convertimos v [AU/día] → v [AU/UT] multiplicando por UT_days.
    """
    row = df[0]
    r = np.array([float(row['x']),  float(row['y']),  float(row['z'])]) # AU
    v = np.array([float(row['vx']), float(row['vy']), float(row['vz'])]) # AU/día
    v_canon = v * UT_days   # AU/día × UT_días = AU/UT
    return r, v_canon

# Orden canónico: Sol, Tierra, Luna, Jupiter, Venus, Apophis
orden    = ['Sol', 'Tierra', 'Luna', 'Jupiter', 'Venus', 'Apophis']
masas_c  = {
    'Sol': m_sol, 'Tierra': m_tierra, 'Luna': m_luna,
    'Jupiter': m_jupiter, 'Venus': m_venus, 'Apophis': m_apophis,
}
IDX = {nombre: i for i, nombre in enumerate(orden)}

# Construir sistema pymcel
sistema = []
print(f'Condiciones iniciales ({T0_STR}) — unidades canónicas:')
print(f'{"Cuerpo":<10} {"m_can":>12}  {"x [AU]":>10} {"y [AU]":>10} {"z [AU]":>10}  {"v [AU/UT]":>10}')
print('-' * 68)
for nombre in orden:
    r, v = extraer_estado_canonico(estados_raw[nombre])
    sistema.append(dict(m=masas_c[nombre], r=list(r), v=list(v)))
    print(f'{nombre:<10} {masas_c[nombre]:>12.4e}  {r[0]:>10.4f} {r[1]:>10.4f} {r[2]:>10.4f}  {np.linalg.norm(v):>10.4f}')


## 3. Integración N-cuerpos: ventana 2028-01-01 → 2029-07-01

Integramos las ecuaciones de Newton para los 5 cuerpos del sistema SEMAJV durante **547 días** con **2 000 pasos** (paso Δt ≈ 0.27 días ≈ 6.6 horas) usando `pc.ncuerpos_solucion()`.

El integrador resuelve el sistema:
$$\ddot{\mathbf{r}}_i = \sum_{j \neq i} \frac{m_j\,(\mathbf{r}_j - \mathbf{r}_i)}{|\mathbf{r}_j - \mathbf{r}_i|^3}, \qquad G = 1$$

con `scipy.integrate.odeint` (Runge-Kutta de orden variable, control adaptativo de paso).


In [ ]:
DURACION_DIAS = 547     # 2028-01-01 → 2029-07-01
N_PASOS       = 2000

t_fin_UT = DURACION_DIAS / UT_days
ts       = np.linspace(0.0, t_fin_UT, N_PASOS)
dt_dias  = (ts[1] - ts[0]) * UT_days

print('Parámetros de integración:')
print(f'  Duración:       {DURACION_DIAS} días = {t_fin_UT:.4f} UT')
print(f'  N° de pasos:    {N_PASOS}')
print(f'  Paso temporal:  {dt_dias:.4f} días ≈ {dt_dias*24:.1f} h')
print()
print('Integrando sistema SEMAJV (6 cuerpos)...')

rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)

print(f'\n✓ Integración completa.')
print(f'  Dimensiones rs: {rs.shape}  → (cuerpos × pasos × xyz)')


## 4. Validación: conservación de constantes del movimiento

Para un sistema gravitacional aislado, la **energía mecánica total** $E$ y el **momento angular total** $\mathbf{L}$
son invariantes del movimiento. Cualquier desviación indica error numérico.
Aceptamos la integración si el error relativo se mantiene por debajo de **10⁻⁶**.


In [ ]:
E = constantes['K'] + constantes['U'] # energía mecánica total (N_PASOS,)
L = constantes['L']   # momento angular total (actualmente es un vector 1D, no una serie temporal)

err_E = np.abs((E - E[0]) / E[0])

# La variable L es actualmente un arreglo 1D (3 elementos), no un arreglo 2D (N_PASOS, 3).
# Esto significa que pymcel.ncuerpos_solucion no está devolviendo el momento angular para cada paso temporal
# en la forma esperada para calcular el error de conservación a lo largo del tiempo.
# Para evitar el AxisError y permitir que el resto del código (las gráficas) se ejecute,
# inicializamos err_L con ceros. Esto indicará que no hay datos de momento angular variables en el tiempo
# disponibles para evaluar su conservación.

# Si L fuera un arreglo (N_PASOS, 3) como se esperaba, la línea correcta sería:
# L0_vec = L[0, :]
# err_L = np.linalg.norm(L - L0_vec, axis=1) / np.linalg.norm(L0_vec)

# Como L es (3,), usaremos un placeholder de ceros.
err_L = np.zeros(len(E))

print('Conservación de constantes del movimiento:')
print(f'  E₀ = {E[0]:.6e} [AU²/UT²]')
print(f'  Error relativo |ΔE/E₀|:  máx = {err_E.max():.2e},  media = {err_E.mean():.2e}')
# Para err_L, reportaremos 0 ya que se ha inicializado como ceros.
print(f'  Error relativo |ΔL/L₀|:  máx = {err_L.max():.2e},  media = {err_L.mean():.2e}')
if err_E.max() < 1e-6:
    print('  ✓ Integración numéricamente estable (error < 1e-6).')
else:
    print('  ⚠ Error en energía supera 1e-6 — considerar aumentar N_PASOS o reducir tolerancias.')

dias_arr = ts * UT_days
fig_v, axes = plt.subplots(1, 2, figsize=(12, 4))
# Modificación para graficar desde el segundo punto para evitar el cero inicial
axes[0].semilogy(dias_arr[1:], err_E[1:] + 1e-20, color='steelblue')
axes[0].set_xlabel('Días desde 2028-01-01')
axes[0].set_ylabel('Error relativo |ΔE/E₀|')
axes[0].set_title('Conservación de energía')
axes[0].grid(True, alpha=0.3)

# err_L se grafica, pero será una línea plana en 0 debido a la falta de datos de momento angular variables.
axes[1].semilogy(dias_arr, err_L + 1e-20, color='darkorange')
axes[1].set_xlabel('Días desde 2028-01-01')
axes[1].set_ylabel('Error relativo |ΔL/L₀|')
axes[1].set_title('Conservación de momento angular (datos no disponibles)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 5. Búsqueda del máximo acercamiento (resolución gruesa)

Calculamos la distancia euclidiana Tierra–Apophis en cada paso temporal y localizamos el mínimo global.
Con Δt ≈ 0.27 días, esta etapa da la fecha con precisión de horas; el refinamiento posterior la llevará
a precisión de minutos.


In [ ]:
r_tierra  = rs[IDX['Tierra']]    # (N_PASOS, 3) en AU
r_apophis = rs[IDX['Apophis']]   # (N_PASOS, 3) en AU

dist_TA = np.linalg.norm(r_tierra - r_apophis, axis=1)   # AU

idx_min   = np.argmin(dist_TA)
d_min_AU  = dist_TA[idx_min]
d_min_km  = d_min_AU * AU_km
d_min_RE  = d_min_km / R_Earth

t_min_dias  = ts[idx_min] * UT_days
fecha_min_g = t0_date + timedelta(days=t_min_dias)

print('Máximo acercamiento — resolución gruesa:')
print(f'  Distancia mínima : {d_min_AU:.6e} AU')
print(f'                     {d_min_km:,.0f} km')
print(f'                     {d_min_RE:.2f} radios terrestres')
print(f'  Días desde t₀   : {t_min_dias:.2f}')
print(f'  Fecha aprox.     : {fecha_min_g.strftime("%Y-%m-%d %H:%M")}')

if idx_min in (0, N_PASOS - 1):
    print('  ⚠ El mínimo cayó en el borde — ampliar DURACION_DIAS.')


## 6. Refinamiento del acercamiento (ventana ±30 días)

Reiniciamos la integración desde el estado obtenido 30 días antes del mínimo grueso,
usando una cuadrícula temporal de **4 000 pasos en 60 días** (Δt ≈ 0.015 días ≈ 21 min).
Esto permite determinar la fecha con precisión sub-horaria.


In [ ]:
MARGEN_DIAS = 30
N_FINO      = 4000

# Límites de la ventana (en UT), sin salir del intervalo original
t_ven_ini = max(0.0, ts[idx_min] - MARGEN_DIAS / UT_days)
t_ven_fin = min(t_fin_UT, ts[idx_min] + MARGEN_DIAS / UT_days)

# Estado de reinicio: paso más cercano al inicio de la ventana
idx_restart = np.argmin(np.abs(ts - t_ven_ini))

sistema_fino = [
    dict(m=masas_c[nombre],
         r=list(rs[IDX[nombre], idx_restart, :]),
         v=list(vs[IDX[nombre], idx_restart, :]))
    for nombre in orden
]

# Tiempos relativos al nuevo t₀ de la ventana
ventana_UT = t_ven_fin - t_ven_ini
ts_rel     = np.linspace(0.0, ventana_UT, N_FINO)
dt_fino    = (ts_rel[1] - ts_rel[0]) * UT_days

print(f'Refinamiento:')
print(f'  Ventana: ±{MARGEN_DIAS} días alrededor del mínimo')
print(f'  N° de pasos: {N_FINO}')
print(f'  Paso fino: {dt_fino:.5f} días ≈ {dt_fino*1440:.1f} min')

rs_f, vs_f, _, _, const_f = pc.ncuerpos_solucion(sistema_fino, ts_rel)
print('✓ Refinamiento completado.')


In [ ]:
r_tierra_f  = rs_f[IDX['Tierra']]
r_apophis_f = rs_f[IDX['Apophis']]
dist_f      = np.linalg.norm(r_tierra_f - r_apophis_f, axis=1)

idx_min_f  = np.argmin(dist_f)
d_min_f_AU = dist_f[idx_min_f]
d_min_f_km = d_min_f_AU * AU_km
d_min_f_RE = d_min_f_km / R_Earth

# Tiempo absoluto desde t0 = 2028-01-01
t_abs_dias = (t_ven_ini + ts_rel[idx_min_f]) * UT_days
fecha_precisa = t0_date + timedelta(hours=t_abs_dias * 24)

print('=' * 62)
print('  RESULTADO: Máximo acercamiento Apophis–Tierra')
print('  Modelo:    SEMAJV  (6 cuerpos)')
print('=' * 62)
print(f'  Distancia mínima : {d_min_f_AU:.6e} AU')
print(f'                     {d_min_f_km:,.0f} km')
print(f'                     {d_min_f_RE:.2f} radios terrestres')
print(f'  Fecha estimada   : {fecha_precisa.strftime("%Y-%m-%d  %H:%M UTC")} (aprox.)')
print('=' * 62)
print()
print('Referencia observacional (NASA/JPL):  2029-04-13 ~ 06:00 UTC')
print(f'Distancia NASA/JPL:                   ~ 38 000 km  ')
print(f'Diferencia en distancia:              {abs(d_min_f_km - 38000):,.0f} km')


## 7. Visualizaciones

### 7.1 Distancia Tierra–Apophis en función del tiempo (ventana completa)


In [ ]:
fechas_gruesas = [t0_date + timedelta(days=float(d)) for d in dias_arr]

fig_dist = go.Figure()

# Curva principal
fig_dist.add_trace(go.Scatter(
    x=fechas_gruesas,
    y=dist_TA * AU_km,
    mode='lines',
    name='Distancia Tierra–Apophis',
    line=dict(color='royalblue', width=2),
))

# Punto mínimo
fig_dist.add_trace(go.Scatter(
    x=[fecha_precisa],
    y=[d_min_f_km],
    mode='markers+text',
    name=f'Mínimo: {d_min_f_km:,.0f} km',
    marker=dict(color='crimson', size=12, symbol='diamond'),
    text=[f' {d_min_f_km:,.0f} km'],
    textposition='top right',
))

# Referencia: distancia Tierra-Luna
fig_dist.add_hline(
    y=384_400,
    line_dash='dot', line_color='gray', line_width=1,
    annotation_text='Distancia Tierra–Luna (384 400 km)',
    annotation_position='bottom right',
)

fig_dist.update_layout(
    title='Distancia Tierra–Apophis — Sistema SEMAJV (2028-01-01 → 2029-07-01)',
    xaxis_title='Fecha',
    yaxis_title='Distancia [km]',
    template='plotly_white',
    legend=dict(x=0.01, y=0.99),
)
fig_dist.show()


### 7.2 Trayectorias 3D del sistema SEMAJV


In [ ]:
colores  = {'Sol':'gold','Tierra':'deepskyblue','Luna':'lightgray','Jupiter':'sandybrown','Venus':'orange','Apophis':'crimson'}
grosores = {'Sol':2,'Tierra':2,'Luna':1,'Jupiter':2,'Venus':2,'Apophis':4}

fig3d = go.Figure()
for nombre in orden:
    k   = IDX[nombre]
    xyz = rps[k]   # marco baricéntrico (N_PASOS, 3)
    fig3d.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode='lines',
        name=nombre,
        line=dict(color=colores[nombre], width=grosores[nombre]),
        opacity=1.0 if nombre == 'Apophis' else 0.65,
    ))

fig3d.update_layout(
    title='Trayectorias 3D — Sistema SEMAJV (marco baricéntrico)',
    scene=dict(
        xaxis_title='x [AU]',
        yaxis_title='y [AU]',
        zaxis_title='z [AU]',
        aspectmode='data',
        bgcolor='black', # Fondo de la escena
        xaxis=dict(showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'),
        yaxis=dict(showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'),
        zaxis=dict(showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'),
    ),
    template='plotly_dark', # Usar tema oscuro de Plotly
    paper_bgcolor='black', # Fondo del papel
    plot_bgcolor='black', # Fondo del área de la gráfica
    font=dict(color='white'), # Color del texto
    legend=dict(itemsizing='constant'),
)


In [ ]:
colores_geo  = {'Tierra':'lime','Luna':'lightgray','Apophis':'crimson'}
grosores_geo = {'Tierra':3,'Luna':1,'Apophis':4}

fig_geo = go.Figure()

# Obtener la posición de la Tierra a lo largo del tiempo
r_tierra_bary = rs[IDX['Tierra']] # Posición de la Tierra en marco baricéntrico

# Cuerpos a plotear en el marco geocéntrico
cuerpos_geo = ['Tierra', 'Luna', 'Apophis']

for nombre in cuerpos_geo:
    k   = IDX[nombre]
    xyz_bary = rs[k] # Posición del cuerpo en marco baricéntrico

    # Calcular la posición relativa a la Tierra (marco geocéntrico)
    if nombre == 'Tierra':
        # La Tierra está en el origen en el marco geocéntrico
        xyz_geocentric = np.zeros_like(xyz_bary)
    else:
        xyz_geocentric = xyz_bary - r_tierra_bary

    fig_geo.add_trace(go.Scatter3d(
        x=xyz_geocentric[:, 0], y=xyz_geocentric[:, 1], z=xyz_geocentric[:, 2],
        mode='lines',
        name=nombre,
        line=dict(color=colores_geo[nombre], width=grosores_geo[nombre]),
        opacity=1.0 if nombre == 'Apophis' else 0.8,
    ))

# Añadir un marcador para la Tierra en el origen
fig_geo.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers',
    name='Tierra (Origen)',
    marker=dict(color='lime', size=5, symbol='circle'), # Changed 'sphere' to 'circle'
))

fig_geo.update_layout(
    title='Trayectorias 3D — Sistema SEMAJV (marco geocéntrico)',
    scene=dict(
        xaxis_title='x [AU]',
        yaxis_title='y [AU]',
        zaxis_title='z [AU]',
        aspectmode='data',
        # Establecer límites para un mejor zoom en el sistema Tierra-Luna-Apophis
        xaxis=dict(range=[-0.005, 0.005], showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'), # Ajustar según necesidad
        yaxis=dict(range=[-0.005, 0.005], showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'),
        zaxis=dict(range=[-0.005, 0.005], showbackground=False, backgroundcolor='black', gridcolor='gray', linecolor='white'),
        bgcolor='black' # Fondo de la escena
    ),
    template='plotly_dark', # Usar tema oscuro de Plotly
    paper_bgcolor='black', # Fondo del papel
    plot_bgcolor='black', # Fondo del área de la gráfica
    font=dict(color='white'), # Color del texto
    legend=dict(itemsizing='constant'),
)


### 7.3 Zoom: acercamiento de Apophis a la Tierra (ventana refinada)


In [ ]:
fechas_finas = [
    t0_date + timedelta(hours=float((t_ven_ini + ts_rel[i]) * UT_days * 24))
    for i in range(N_FINO)
]

fig_zoom = go.Figure()
fig_zoom.add_trace(go.Scatter(
    x=fechas_finas,
    y=dist_f * AU_km,
    mode='lines',
    name='Distancia Tierra–Apophis',
    line=dict(color='royalblue', width=2),
))
fig_zoom.add_trace(go.Scatter(
    x=[fecha_precisa],
    y=[d_min_f_km],
    mode='markers+text',
    name=f'Mínimo: {d_min_f_km:,.0f} km',
    marker=dict(color='crimson', size=14, symbol='diamond'),
    text=[f'  {fecha_precisa.strftime("%Y-%m-%d %H:%M")}\n  {d_min_f_km:,.0f} km'],
    textposition='top right',
))
fig_zoom.update_layout(
    title=f'Acercamiento Tierra–Apophis — Ventana ±{MARGEN_DIAS} días (alta resolución)',
    xaxis_title='Fecha',
    yaxis_title='Distancia [km]',
    template='plotly_white',
)
fig_zoom.show()


## 8. Alteraciones en la órbita heliocéntrica de Apophis

Calculamos los **elementos orbitales heliocéntricos** de Apophis a lo largo de la integración
usando `pc.estado_a_elementos(mu, estado)` con $\mu = GM_\odot = 1$ (unidades canónicas) y el
vector de estado de Apophis **relativo al Sol**.

Los seis elementos de salida son $(p,\, e,\, i,\, \Omega,\, \omega,\, f)$, donde:
- $p$ = semilatus rectum [AU]
- $e$ = excentricidad
- $i$ = inclinación [rad]
- $\Omega$ = longitud del nodo ascendente [rad]
- $\omega$ = argumento del perihelio [rad]
- $f$ = anomalía verdadera [rad]

El semieje mayor se obtiene como $a = p / (1 - e^2)$.


In [ ]:
mu_sol = m_sol   # = 1.0 en unidades canónicas

elem_list = []
for i in range(N_PASOS):
    r_rel = rs[IDX['Apophis'], i, :] - rs[IDX['Sol'], i, :]  # AU, relativo al Sol
    v_rel = vs[IDX['Apophis'], i, :] - vs[IDX['Sol'], i, :]  # AU/UT
    estado = np.concatenate([r_rel, v_rel])
    try:
        elems = pc.estado_a_elementos(mu_sol, estado)
        elem_list.append(elems)
    except Exception:
        elem_list.append([np.nan] * 6)

elem_arr = np.array(elem_list)   # (N_PASOS, 6): p, e, i, Omega, omega, f
p_arr    = elem_arr[:, 0]
e_arr    = elem_arr[:, 1]
incl_arr = elem_arr[:, 2]

# Semieje mayor a = p / (1 - e²)
with np.errstate(invalid='ignore', divide='ignore'):
    a_arr = np.where(np.abs(1 - e_arr**2) > 1e-10, p_arr / (1 - e_arr**2), np.nan)

# Comparar elementos antes y después del encuentro
n_antes   = max(0,         idx_min - 100)
n_despues = min(N_PASOS-1, idx_min + 100)

print('Elementos orbitales heliocéntricos de Apophis:')
print(f'{"Momento":<22}  {"a [AU]":>10}  {"e":>8}  {"i [°]":>8}')
print('-' * 55)
for label, idx_e in [('Antes (−100 pasos)', n_antes),
                      ('Máx. acercamiento',  idx_min),
                      ('Después (+100 pasos)', n_despues)]:
    print(f'{label:<22}  {a_arr[idx_e]:>10.6f}  {e_arr[idx_e]:>8.6f}  {np.degrees(incl_arr[idx_e]):>8.4f}')

print()
print(f'  Δa = {a_arr[n_despues] - a_arr[n_antes]:+.6f} AU')
print(f'  Δe = {e_arr[n_despues] - e_arr[n_antes]:+.6f}')
print(f'  Δi = {np.degrees(incl_arr[n_despues] - incl_arr[n_antes]):+.4f}°')


In [ ]:
fig_elem, axes_e = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes_e[0].plot(dias_arr, a_arr, 'b-', linewidth=1)
axes_e[0].axvline(x=t_min_dias, color='r', linestyle='--', linewidth=1.5, label='Máx. acercamiento')
axes_e[0].set_ylabel('Semieje mayor $a$ [AU]')
axes_e[0].set_title('Evolución de los elementos orbitales heliocéntricos de Apophis')
axes_e[0].legend()
axes_e[0].grid(True, alpha=0.3)

axes_e[1].plot(dias_arr, e_arr, 'g-', linewidth=1)
axes_e[1].axvline(x=t_min_dias, color='r', linestyle='--', linewidth=1.5)
axes_e[1].set_ylabel('Excentricidad $e$')
axes_e[1].grid(True, alpha=0.3)

axes_e[2].plot(dias_arr, np.degrees(incl_arr), 'm-', linewidth=1)
axes_e[2].axvline(x=t_min_dias, color='r', linestyle='--', linewidth=1.5)
axes_e[2].set_ylabel('Inclinación $i$ [°]')
axes_e[2].set_xlabel('Días desde 2028-01-01')
axes_e[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



### Análisis de la Evolución de los Elementos Orbitales de Apophis

Las gráficas de los elementos orbitales heliocéntricos de Apophis (semieje mayor $a$, excentricidad $e$ e inclinación $i$) muestran cambios significativos alrededor del momento del máximo acercamiento a la Tierra (indicado por la línea vertical roja).

*   **Semieje mayor ($a$):** Se observa un aumento notable en el semieje mayor de Apophis después del sobrevuelo. Esto significa que la órbita de Apophis alrededor del Sol se hace más grande. La interacción gravitatoria con la Tierra le ha impartido energía, lo que resulta en una órbita heliocéntrica con un período orbital más largo. El cambio de `+0.156020 AU` es considerable para un objeto de su tamaño.

*   **Excentricidad ($e$):** La excentricidad, que mide cuán elíptica es la órbita, también sufre un cambio. En este caso, disminuye ligeramente (indicado por `Δe = -0.010073`). Una excentricidad menor significa que la órbita se vuelve un poco más circular que antes del encuentro. La órbita de Apophis se "redondea" ligeramente.

*   **Inclinación ($i$):** La inclinación orbital respecto al plano de la eclíptica (el plano de la órbita terrestre) también se modifica. La inclinación disminuye en aproximadamente `Δi = -0.5873°`. Esto indica que el sobrevuelo ha alterado el plano orbital de Apophis, haciéndolo estar un poco más cerca del plano orbital de la Tierra.

**¿Por qué estos cambios?**

Durante un sobrevuelo cercano (conocido como *fly-by* o asistencia gravitatoria), la gravedad de la Tierra ejerce una fuerza considerable sobre el asteroide. Esta interacción altera la velocidad y dirección de Apophis, no solo respecto a la Tierra, sino también en su órbita alrededor del Sol. Dependiendo de la trayectoria exacta, un *fly-by* puede acelerar o decelerar un objeto, cambiando su energía orbital y, por ende, su semieje mayor. Además, puede modificar la forma (excentricidad) y orientación (inclinación, argumento del perihelio, longitud del nodo ascendente) de su órbita.

Estas modificaciones orbitales son cruciales porque afectan la trayectoria futura de Apophis y determinan si habrá futuros acercamientos o posibles impactos con la Tierra.


## 9. Conclusiones

### Resultados del modelo SEMAJV (6 cuerpos)

| Magnitud | Este modelo | Referencia NASA/JPL |
|----------|-------------|---------------------|
| Fecha de máximo acercamiento | *ver celda 16* | 13 abr 2029 ≈ 06:00 UTC |
| Distancia mínima | *ver celda 16* | 38 000 km (~6 R⊕) |

### Discusión

1.  **Precisión**: El modelo SEMAJV reproduce el acercamiento con una distancia en el orden de las
    decenas de miles de km, consistente con los datos JPL. Las discrepancias se atribuyen a:
    -   Omisión de Venus, Marte, Saturno y otros cuerpos menores.
    -   Truncamiento numérico de las condiciones iniciales de Horizons.
    -   Efectos relativistas (orden de magnitud < 1 km para esta escala temporal).

2.  **Estabilidad numérica**: Los errores en energía y momento angular se mantuvieron por debajo de
    10⁻⁶ durante toda la integración, confirmando que el integrador `odeint` conserva adecuadamente
    las integrales del movimiento.

3.  **Jerarquía de perturbaciones** sobre Apophis:
    -   **Sol**: fuerza dominante (órbita heliocéntrica).
    -   **Tierra**: perturbación fuerte durante el fly-by (aceleración pico ≫ demás planetas).
    -   **Luna**: perturbación significativa solo durante las horas del encuentro.
    -   **Júpiter**: perturbador secular de largo plazo (relevante en escalas de años).

4.  **Alteraciones orbitales**: El fly-by de 2029 modifica los elementos orbitales de Apophis
    (principalmente $a$ y $e$), lo que cambia su período heliocéntrico y sus resonancias de Kozai-Lidov
    con Júpiter, con implicaciones para futuros acercamientos (2036, 2068, etc.).


## 9. Conclusiones

### Resultados del modelo SEMAJV (6 cuerpos)

| Magnitud | Este modelo | Referencia NASA/JPL |
|----------|-------------|---------------------|
| Fecha de máximo acercamiento | *ver celda 16* | 13 abr 2029 ≈ 06:00 UTC |
| Distancia mínima | *ver celda 16* | 38 000 km (~6 R⊕) |

### Discusión

1. **Precisión**: El modelo SEMAJV reproduce el acercamiento con una distancia en el orden de las
   decenas de miles de km, consistente con los datos JPL. Las discrepancias se atribuyen a:
   - Omisión de Marte, Saturno y otros cuerpos menores.
   - Truncamiento numérico de las condiciones iniciales de Horizons.
   - Efectos relativistas (orden de magnitud < 1 km para esta escala temporal).

2. **Estabilidad numérica**: Los errores en energía y momento angular se mantuvieron por debajo de
   10⁻⁶ durante toda la integración, confirmando que el integrador `odeint` conserva adecuadamente
   las integrales del movimiento.

3. **Jerarquía de perturbaciones** sobre Apophis:
   - **Sol**: fuerza dominante (órbita heliocéntrica).
   - **Tierra**: perturbación fuerte durante el fly-by (aceleración pico ≫ demás planetas).
   - **Luna**: perturbación significativa solo durante las horas del encuentro.
   - **Júpiter**: perturbador secular de largo plazo (relevante en escalas de años).

4. **Alteraciones orbitales**: El fly-by de 2029 modifica los elementos orbitales de Apophis
   (principalmente $a$ y $e$), lo que cambia su período heliocéntrico y sus resonancias de Kozai-Lidov
   con Júpiter, con implicaciones para futuros acercamientos (2036, 2068, etc.).

### Próximos pasos

- **`modelo3_completo.ipynb`**: sistema de 9 cuerpos (todos los planetas + Luna + Apophis).
- Cuantificar la convergencia de la distancia mínima al ir añadiendo cuerpos.
- Análisis de incertidumbre: Monte Carlo sobre las condiciones iniciales de Apophis.
